In [ ]:
import os 
import sys
sys.path.append('../src')

from preprocessing import *

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)

In [ ]:
ca, lab = create_vitals_df(dfs)
med = create_medication_df(dfs)

In [ ]:
lab[lab['description'] == 'AlbuminKSU']['unit'].value_counts()
x =lab[lab['description'] == 'AlbuminKSU']

In [ ]:
ts_data = create_ts_data(ca, vitals_lab=lab, medication=med, merge_lab=True, merge_med=False)
ts_data

In [ ]:
ca.columns

In [ ]:
print(len(dfs['lab_cohort']))
dfs['lab_cohort']['Bezeichnung'].value_counts().head(15)

In [ ]:
# create vitals from clinical assessments 
print(dfs['clinical_assessment'].columns)
print(dfs['clinical_assessment'].count())
# no data points for Infektion, Zielwerte, Uhrzeit

In [ ]:
vitals_ca = dfs['clinical_assessment'][['PatientID', 'TransplantationID', 'OPDtime', 'Blutdruck_systolisch', 'Blutdruck_diastolisch', 'Gewicht', 'Urinvolumen', 'Herzfrequenz', 'Temperatur', 'Diuresezeit']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'OPDtime': 'rel_days', # date of assessment in days after transplantation
            'Blutdruck_systolisch': 'bp_sys',
            'Blutdruck_diastolisch': 'bp_dia',
            'Gewicht': 'weight',
            'Urinvolumen': 'urine_volume',  # lower urine volume indicates issues with kidney function
            'Herzfrequenz': 'hr',
            'Temperatur': 'temperature', 
            'Diuresezeit': 'diuresis_time',  # duration over which urine output is measured
        }
    )

print(f"Unique patients in clinical assessments: {vitals_ca['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
vitals_ca = vitals_ca.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Unique patients in clinical assessments: {vitals_ca['patient_id'].nunique()}")
print(f"Average entries per patient {len(vitals_ca) / vitals_ca['patient_id'].nunique()}")

In [ ]:
vitals_ca

In [ ]:
print(dfs['lab_cohort'].columns)
print(dfs['lab_cohort'].count())

In [ ]:
vitals_lab = dfs['lab_cohort'][['PatientID', 'TransplantationID', 'Labtime', 'Bezeichnung', 'Wert', 'Einheit']].rename(
        columns={
            'PatientID': 'patient_id',
            'TransplantationID': 'transplant_id',
            'Labtime': 'rel_days', # date of assessment in days after transplantation
            'Bezeichnung': 'description',
            'Wert' : 'value',
            'Einheit': 'unit',
        }
    )

print(f"Unique patients in lab df: {vitals_lab['patient_id'].nunique()}")
print('Removing patients that are not in static_df')
vitals_lab = vitals_lab.merge(
    static_df[['patient_id', 'transplant_id']],
    how='inner',  # Inner join to keep only matching entries
    on=['patient_id', 'transplant_id']
)

print(f"Unique patients in lab df: {vitals_lab['patient_id'].nunique()}")
print(f"Average entries per patient {len(vitals_lab) / vitals_lab['patient_id'].nunique()}")